In [1]:
using Random
using LinearAlgebra
using ProgressMeter
using JLD2
using MosekTools
using Ket
using JuMP

using ppt2

In [ ]:
# Load precomputed forms
files_4x4 = [
    "../pncp_forms_4x4.jld2",
]

forms_4x4 = vcat([jldopen(file) do file
    vcat([file[k] for k in keys(file)]...)
end for file in files_4x4]...)

forms_4x4[1]

16×16 Matrix{Float64}:
  0.250562   -0.185018    0.595791   -0.0421582  …   0.0121876    -0.0844437
 -0.185018    0.242161   -0.725767    0.0369451      0.11218       0.0838693
  0.595791   -0.725767    2.45581    -0.100628       0.208687     -0.4419
 -0.0421582   0.0369451  -0.100628    0.0861272     -0.4419        0.0523178
 -0.260352    0.243012   -0.959363    0.239431      -0.0108145     0.172371
  0.243012   -0.29567     1.00193    -0.242058   …  -0.301265     -0.206355
 -0.959363    1.00193    -3.15661     0.833063      -0.661692      0.647687
  0.239431   -0.242058    0.833063    0.0179045      0.647687     -0.389412
  0.121636   -0.0532369   0.110013   -0.117369      -0.0413898    -0.0591179
 -0.0532369  -0.0175779   0.203736    0.0851487     -0.025026      0.0648484
  0.110013    0.203736   -1.24314    -0.247287   …  -0.250823      0.262551
 -0.117369    0.0851487  -0.247287   -0.0722558      0.262551      0.111234
 -0.124737    0.0615256   0.0121876  -0.0844437     -0.0007743

In [ ]:
function is_pos(form, n::Int, m::Int; attempts=100000, tol=1e-6)
    mi = Inf
    for _ in 1:attempts
        x = randn(n)
        y = randn(m)
        xy = kron(x, y)

        mi = min(mi, xy' * form * xy)
        if mi < -tol
            return mi, x, y
        end
    end
    return mi, nothing, nothing
end

function min_xy_form(
    form, n::Int, m::Int; restarts::Int=16, max_iter::Int=40,
    tol::Float64=1e-8, verbose::Bool=false
)
    Q = Matrix(Hermitian(form))

    best_val = Inf
    best_x = zeros(n)
    best_y = zeros(m)

    # Two small Mosek SDPs are alternated: optimize X with Y fixed, then Y with X fixed.
    modelX = Model(Mosek.Optimizer)
    modelY = Model(Mosek.Optimizer)
    set_silent(modelX)
    set_silent(modelY)

    @variable(modelX, X[1:n, 1:n], PSD)
    @constraint(modelX, sum(X[i, i] for i in 1:n) == 1.0)

    @variable(modelY, Y[1:m, 1:m], PSD)
    @constraint(modelY, sum(Y[j, j] for j in 1:m) == 1.0)

    for r in 1:restarts
        y0 = randn(m)
        y0 /= norm(y0)
        Ycur = y0 * y0'
        Xcur = nothing

        prev = Inf
        for it in 1:max_iter
            A = zeros(n, n)
            for i in 1:n, k in 1:n
                s = 0.0
                for j in 1:m, l in 1:m
                    s += Q[(i - 1) * m + j, (k - 1) * m + l] * Ycur[j, l]
                end
                A[i, k] = s
            end
            @objective(modelX, Min, sum(A[i, k] * X[i, k] for i in 1:n, k in 1:n))
            optimize!(modelX)
            Xcur = Symmetric(value.(X))

            B = zeros(m, m)
            for j in 1:m, l in 1:m
                s = 0.0
                for i in 1:n, k in 1:n
                    s += Q[(i - 1) * m + j, (k - 1) * m + l] * Xcur[i, k]
                end
                B[j, l] = s
            end
            @objective(modelY, Min, sum(B[j, l] * Y[j, l] for j in 1:m, l in 1:m))
            optimize!(modelY)
            Ycur = Symmetric(value.(Y))

            val = sum(Q[(i - 1) * m + j, (k - 1) * m + l] * Xcur[i, k] * Ycur[j, l]
                      for i in 1:n, j in 1:m, k in 1:n, l in 1:m)

            if verbose
                println("restart=$r iter=$it value=$val")
            end

            if abs(prev - val) < tol
                break
            end
            prev = val
        end

        ex = eigen(Hermitian(Xcur))
        ey = eigen(Hermitian(Ycur))
        x = ex.vectors[:, argmax(ex.values)]
        y = ey.vectors[:, argmax(ey.values)]

        xy = kron(x, y)
        val_pure = real(xy' * Q * xy)

        if val_pure < best_val
            best_val = val_pure
            best_x = x
            best_y = y
        end
    end

    return best_val, best_x, best_y
end

function diagnostics(ppt, wit, n::Int, m::Int)
    println("Eigvals of channel:")
    println(eigvals(ppt))
    cpp = Hermitian(ampliation(ppt, ppt, n, m))
    println("Eigvals of composite:")
    println(eigvals(cpp))
    println("Eigvals of witness:")
    println(eigvals(wit))
    println("Positivity of witness map:")
    println(is_pos(wit, n, m, attempts=10000000, tol=1e-6))
    println("Expectation value of the witness on the CPP:")
    println(tr(wit * cpp))
end

diagnostics (generic function with 1 method)

In [4]:
open("out.txt","a") do io
    i = 1
    @showprogress for form in forms_4x4
        println(io, "Form: $i")
        i += 1
        v, x, y = is_pos(form, 4, 4, attempts=1e6, tol=0.0)
        println(io, "Minimum: $v")
        if v < 0
            println(io, "x=$x")
            println(io, "y=$y")
        end
        v, x, y = min_xy_form(form, 4, 4, restarts=16, max_iter=40, tol=0.0, verbose=false)
        println(io, "Minimum: $v")
        if v < 0
            println(io, "x=$x")
            println(io, "y=$y")
        end
    end
end

Progress: 100%|█████████████████████████████████████████| Time: 0:30:20
